# 01. Data Preprocessing

This notebook performs the process of reading raw data `Online Retail.xlsx`, cleaning it, and converting it to the basket format required for the ECLAT algorithm.

The algorithm internally uses `StockCode` to ensure consistency, but the baskets are stored with product names (`Description`) for better readability. A `StockCode -> Description` mapping table is created separately for reference if needed.

In [ ]:
# Import necessary libraries
import pandas as pd
import os

## 1. Read raw data


In [ ]:
# Set working directory to project root if running directly from notebooks/ folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

raw_data_path = 'data/raw/Online Retail.xlsx'
print(f"Reading data from: {raw_data_path} ...")

df = pd.read_excel(raw_data_path)
print(f"Raw data shape: {df.shape[0]} records, {df.shape[1]} columns")
df.head()

## 2. Data Cleaning
- Drop rows with missing `Description`
- Convert `Description` to string and remove redundant whitespaces
- Drop rows with missing invoice number (`InvoiceNo` is null)
- Remove cancelled invoices (contains 'C' in `InvoiceNo`)
- Keep only transactions with `Quantity > 0`
- Remove special `StockCode`s that are not actual products by using regex

In [ ]:
# Drop rows with missing Description
df.dropna(axis=0, subset=['Description'], inplace=True)

# Convert Description to string and strip whitespaces
df['Description'] = df['Description'].astype(str).str.strip()

# Drop rows with missing InvoiceNo
df.dropna(axis=0, subset=['InvoiceNo'], inplace=True)
df['InvoiceNo'] = df['InvoiceNo'].astype('str')

# Remove cancelled transactions (InvoiceNo contains 'C')
df = df[~df['InvoiceNo'].str.contains('C')]

# Keep only transactions with Quantity > 0
df = df[df['Quantity'] > 0]

# Convert StockCode to string and keep only valid product codes using regex
# Valid codes: 5 digits optionally followed by 1 letter (e.g., '85123', '85123A')
df['StockCode'] = df['StockCode'].astype('str')
df = df[df['StockCode'].str.match(r'^\d{5}[a-zA-Z]?$')]

print(f"Number of transaction records after cleaning: {df.shape[0]}")
print(f"Number of unique StockCodes: {df['StockCode'].nunique()}")
print(f"Number of unique InvoiceNos: {df['InvoiceNo'].nunique()}")

## 3. Create StockCode -> Description mapping table
This table is used to look up product names by code when needed.

In [ ]:
mapping = df.groupby('StockCode')['Description'].first().reset_index()

mapping_path = 'data/processed/stockcode_description_map.csv'
mapping.to_csv(mapping_path, index=False)
print(f"Mapping table saved to: {mapping_path}")
print(f"Total number of product codes in mapping table: {mapping.shape[0]}")
mapping.head(10)

## 4. Save cleaned transaction data
Ready for use by other algorithms later.

In [ ]:
processed_transactions_path = 'data/processed/transactions_cleaned.csv'
df.to_csv(processed_transactions_path, index=False)
print(f"Successfully saved cleaned transaction table to: {processed_transactions_path}")
df.head()

## 5. Convert to Basket Format
Group products (by `Description`) purchased together in the same invoice into a list, remove duplicates, and sort them to ensure consistency. This is the required input format for the ECLAT algorithm.

In [ ]:
# Group by transaction, get a unique list of product names for each invoice
baskets = df.groupby('InvoiceNo')['Description'].apply(
    lambda x: ','.join(sorted(set(x)))
).reset_index()
baskets.columns = ['InvoiceNo', 'Items']

baskets_path = 'data/processed/baskets.csv'
baskets.to_csv(baskets_path, index=False)

print(f"Successfully saved basket file to: {baskets_path}")
print(f"Total number of baskets: {baskets.shape[0]}")
baskets.head()